# PPO on LunarLander-v3

A Proximal Policy Optimization (PPO) agent learns to land the lunar module on its own, by trial and error.
The environment is considered solved when the agent reaches a mean score of at least 200 over 100 consecutive episodes, played without exploration.

The notebook reads from top to bottom and must run with *Restart & Run All*.

1. Setup
2. Environment
3. Random baseline
4. Actor and critic
5. Rollout storage and advantages
6. PPO update
7. Training
8. Results
9. Experiments
10. Comparison with DQN
11. Conclusion

## 1. Setup

In [9]:
import random
from collections import Counter

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


### Reproducibility

Several libraries draw random numbers in this notebook, and each one has its own generator:

| Library | Used for |
|---|---|
| `random` | the random baseline agent |
| `numpy` | shuffling and statistics helpers |
| `torch` | the initial weights of the networks and the sampling of actions from the policy |

Seeding only one of them would leave the others unpredictable, so `set_seed` fixes the three at once.
It is a function rather than three lines at the top of the notebook because section 8 trains the agent with several seeds: the generators must be reset before each run.

The environment has a generator of its own. It is seeded separately in section 2, through `env.reset(seed=...)`.

In [ ]:
def set_seed(*, seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


SEED = 42
set_seed(seed=SEED)

## 2. Environment

LunarLander-v3 is a 2D landing task simulated with Box2D physics. Gymnasium takes care of the physics, the rewards and the rendering, so this notebook only deals with the agent.

The environment is created without a `render_mode`. Drawing every frame would slow training down a lot, and the agent does not need images: it only reads the observation vector. The display window is opened by `model.py` only, to watch a trained agent.

### Observations

At each step, the agent receives a vector of 8 values:

| Index | Name | Description |
|---|---|---|
| 0 | `x` | horizontal position (the landing pad is at 0, the screen edges at ±1) |
| 1 | `y` | vertical position |
| 2 | `v_x` | horizontal velocity |
| 3 | `v_y` | vertical velocity |
| 4 | `theta` | angle of the lander |
| 5 | `theta_dot` | angular velocity |
| 6 | `left_leg_contact` | 1 if the left leg touches the ground, 0 otherwise |
| 7 | `right_leg_contact` | 1 if the right leg touches the ground, 0 otherwise |

### Actions

The agent picks one of 4 discrete actions:

| Action | Effect |
|---|---|
| 0 | do nothing |
| 1 | fire the left orientation engine |
| 2 | fire the main engine |
| 3 | fire the right orientation engine |

### End of an episode

| Flag | Cause | Is there a future? |
|---|---|---|
| `terminated` | the body touched the ground (crash), the lander left the viewport, or it came to rest | no |
| `truncated` | the episode reached the limit of 1000 steps | yes, the lander could have kept going |

The two flags are kept apart everywhere in this notebook. Section 5 uses the difference: after a truncation, the critic estimates the rewards the agent would still have collected.

### Seeding and dimensions

`env.reset(seed=SEED)` seeds the environment generator once. The following resets are called without a seed and continue the same sequence, so the episodes are reproducible without replaying the same episode over and over. `env.action_space.seed(SEED)` seeds `env.action_space.sample()` in the same way.

The size of the observation and the number of actions are read from `env.observation_space` and `env.action_space` instead of being hard coded.

In [11]:
env = gym.make("LunarLander-v3")
env.action_space.seed(seed=SEED)
state, _ = env.reset(seed=SEED)
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print("State dimension:", state_dim)
print("Number of actions:", n_actions)

State dimension: 8
Number of actions: 4
